# Gridworld A/B z oryginalnego kursu

Zapis modelu P[s][a] jest wspólny z zadaniem o robocie. Stan to numer pola na planszy 5 × 5.
Numer rośnie w prawo i w dół, od zera. Akcje mają kolejność: góra, prawo, dół, lewo.

- Z A = (0, 1) dowolna akcja prowadzi do A′ = (4, 1) i daje +10.
- Z B = (0, 3) dowolna akcja prowadzi do B′ = (2, 3) i daje +5.
- Z pozostałych pól ruch wewnątrz planszy daje 0. Próba wyjścia zostawia agenta na miejscu i daje −1.
- Proces trwa po każdym przejściu. Nagrodę z A i B można otrzymać ponownie.

[Oryginalny notebook Ch04](https://github.com/zussini/kursy/blob/170b201b2fe8fd1300d2ceb3afffab7cb823c020/reinforcement_learning/notebooks/Ch04_Dynamic_Programming_full.ipynb)

**Zanim uruchomisz.** Przewidź wpisy dla A i B po komendzie „prawo”. Co zwróci P na górnej krawędzi po komendzie „góra”? Jakie znaczenie ma ostatnie pole każdej krotki?

In [ ]:
from math import isclose

WIDTH = 5
ACTIONS = ("góra", "prawo", "dół", "lewo")
MOVES = ((-1, 0), (0, 1), (1, 0), (0, -1))
SPECIAL = {(0, 1): ((4, 1), 10), (0, 3): ((2, 3), 5)}

def state(row, col):
    return row * WIDTH + col

def make_model():
    model = {}
    for s in range(WIDTH ** 2):
        row, col = divmod(s, WIDTH)
        model[s] = {}
        for a, (dr, dc) in enumerate(MOVES):
            if (row, col) in SPECIAL:
                (rr, cc), reward = SPECIAL[row, col]
            else:
                rr, cc = row + dr, col + dc
                reward = 0
                if not (0 <= rr < WIDTH and 0 <= cc < WIDTH):
                    rr, cc = row, col
                    reward = -1
            model[s][a] = [(1.0, state(rr, cc), reward, False)]
    return model

P = make_model()
A, B = state(0, 1), state(0, 3)
for a in range(len(ACTIONS)):
    assert P[A][a] == [(1.0, state(4, 1), 10, False)]
    assert P[B][a] == [(1.0, state(2, 3), 5, False)]
assert P[0][0] == [(1.0, 0, -1, False)]
print("P[A][prawo] =", P[A][1])
print("P[B][prawo] =", P[B][1])
print("P[0][góra] =", P[0][0])

## Jedna gałąź, dwa sposoby wyboru

Każda gałąź wnosi \(p[r+\gamma V(s')]\). Znacznik końca to zawsze False, więc dalsza wartość wchodzi do rachunku.

Polityka jednostajna waży wszystkie cztery komendy równo. Iteracja wartości wybiera maksimum z czterech wartości akcji. Oba rachunki w jednym przebiegu czytają z poprzedniej mapy.

**Ćwiczenie.** Zacznij od \(V_0=0\) i \(\gamma=0{,}9\). Policz \(V_1(A)\), \(V_1(B)\), a potem \(V_2(0,0)\) w trybie maksimum. Jak zmieni się wynik dla polityki jednostajnej? Przed drugim przebiegiem oblicz \(V_1\) tylko w sąsiedztwie \((0,0)\).

In [ ]:
def q_value(s, a, values, gamma):
    return sum(p * (reward + (0 if done else gamma * values[next_s]))
               for p, next_s, reward, done in P[s][a])

def action_values(s, values, gamma):
    return [q_value(s, a, values, gamma) for a in P[s]]

def uniform(q):
    return sum(q) / len(q)

def sweep(values, gamma, combine):
    return [combine(action_values(s, values, gamma)) for s in P]

def print_map(values):
    for row in range(WIDTH):
        print("  ".join(f"{values[state(row, col)]:7.3f}" for col in range(WIDTH)))

gamma = 0.9
initial = [0.0] * len(P)
for name, rule in (("jednostajna", uniform), ("maksimum", max)):
    first = sweep(initial, gamma, rule)
    second = sweep(first, gamma, rule)
    print(f"{name}: V1(A)={first[A]:.3f}, V1(B)={first[B]:.3f}, V2(0,0)={second[0]:.5f}")
    print_map(second)
    assert (first[A], first[B]) == (10, 5)
    expected = 1.46875 if rule is uniform else 9
    assert isclose(second[0], expected)

## Po wielu iteracjach

W A agent dostaje +10, po czym trafia do A′. Jeśli wróci bezpośrednio czterema ruchami z A′ do A, nagroda +10 powtarza się co pięć ruchów. Przy \(\gamma=0{,}9\) wartość tej reguły od A wynosi \(10/(1-0{,}9^5)\). Uruchom komórkę i porównaj z maksimum. Czy różnica wychodzi poza dokładność obliczeń?

Zmień jeden parametr. Odczytaj co najmniej jedną zmianę wartości, której potrafisz wskazać przyczynę w modelu.

In [ ]:
def solve(gamma, rule, tolerance=1e-10):
    values = [0.0] * len(P)
    for iterations in range(1, 20_001):
        new_values = sweep(values, gamma, rule)
        change = max(abs(x - y) for x, y in zip(values, new_values))
        if change < tolerance:
            return new_values, iterations
        values = new_values
    raise RuntimeError("Nie osiągnięto zadanej tolerancji.")

v_uniform, n_uniform = solve(gamma, uniform)
v_best, n_best = solve(gamma, max)
for label, values, count in (("Losowa", v_uniform, n_uniform), ("Optymalna", v_best, n_best)):
    print(f"{label}. V(0,0)={values[0]:.4f}. Przebiegi: {count}.")
    print_map(values)

cycle_value = 10 / (1 - gamma ** 5)
print(f"Wynik cyklu A: {cycle_value:.5f}, V*(A): {v_best[A]:.5f}")
assert isclose(v_best[A], cycle_value, rel_tol=1e-8)

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Opcjonalny Matplotlib jest niedostępny. Mapy liczbowe są powyżej, a kolory w HTML.")
else:
    lo = min(v_uniform + v_best)
    hi = max(v_uniform + v_best)
    fig, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)
    for axis, values, title in zip(axes, (v_uniform, v_best), ("Polityka jednostajna", "Wartości optymalne")):
        matrix = [values[i:i+WIDTH] for i in range(0, len(values), WIDTH)]
        image = axis.imshow(matrix, vmin=lo, vmax=hi, cmap="viridis")
        axis.set_title(title)
        axis.set(xticks=range(WIDTH), yticks=range(WIDTH), xlabel="Kolumna", ylabel="Wiersz")
        for label, (row, col) in (("A", (0,1)), ("B", (0,3)), ("A′", (4,1)), ("B′", (2,3))):
            axis.text(col, row, label, color="white", ha="center", va="center",
                      bbox={"facecolor": "#132e35", "alpha": 0.85, "pad": 2, "edgecolor": "none"})
    fig.colorbar(image, ax=axes, label="Oczekiwany zdyskontowany zwrot")
    plt.show()

## Co dalej?

- **Robot.** Jedna komenda może mieć kilka gałęzi. W Gridworldzie A/B mamy dokładnie jedną.
- **FrozenLake.** Wejście do celu lub dziury zamyka epizod. Poślizg znów tworzy kilka wyników tej samej komendy.
- **MC i TD.** Do rachunku dostajemy obserwację z ruchu albo przebiegu. Pełne P zostaje do obliczenia punktu odniesienia.

W oryginalnym Ch04 większą planszę 10 × 10 wykorzystano jako mini-projekt. Możesz przenieść tę samą pętlę i porównać mapy po zmianie jednego parametru.